# master9 — Parametric inertial particle PINN (GPU)

این نسخه با نسخهٔ قبلیِ کنترل نظارت‌شده فرق دارد: **هیچ برچسب فرمان و هیچ مسیر حل‌شده‌ای وارد آموزش نمی‌شود.** آموزش از باقیماندهٔ معادلات سیال و ذره است؛ حل مستقل Radau فقط بعد از آموزش اجرا می‌شود.

ورودی‌ها: رینولدز، سرعت متوسط سیال، چگالی سیال، محل رهایش، دو مؤلفهٔ سرعت اولیهٔ ذره و تحریک مغناطیسی. ویسکوزیته از `eta=rho*U*(2R)/Re` به دست می‌آید و ترکیب نامعتبر رد می‌شود.

محدوده‌ها: eta=0.001..0.006 Pa.s، U=0.005..0.03 m/s، rho=950..1100 kg/m³، X0=-0.7..0.7، Z0=-2..1.7، vx0=±0.001 m/s، vz0=0..0.06 m/s، u=±0.5. شعاع ذره 500nm و کانال 75µm ثابت‌اند.

معادلات: جریان کاملاً توسعه‌یافتهٔ نیوتنی در کانال استوانه‌ای، میدان تحلیلی دو منبع آینه‌ای و حرکت ذره با درگ استوکس و جرم افزودهٔ نصف جرم سیال جابه‌جاشده. گرانش، نیروی تاریخچهٔ Basset، برهم‌کنش دوطرفه و نویز حسگر لحاظ نشده‌اند. گذرای زیرمیکروثانیه‌ای فقط در محدودهٔ این مدل ساده اعتبارسنجی می‌شود.

ساختار شبکه شرط مکان و سرعت اولیه را دقیقاً اعمال می‌کند و لایهٔ گذرای نمایی را از بخش کند حرکت جدا می‌کند. PINN نگاشت گام زمانی کوتاه را می‌آموزد و برای مسیر بلند تکرار می‌شود. کنترل فرمان، حل معکوس مقید با همین PINN است، نه شبکهٔ دارای برچسب جواب.


In [ ]:
import os,sys,io,base64,zipfile,json,time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'GPU was requested but is unavailable; do not silently train on CPU.'
ROOT=Path.cwd()/'master9_run'
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as z:z.extractall(ROOT)
sys.path.insert(0,str(ROOT))
OUTPUT=ROOT/'outputs'
print('GPU:',torch.cuda.get_device_name(0))


## وزن‌های آموزش‌دیده روی GPU؛ فقط باقیماندهٔ معادلات
نسخهٔ ۱ روی Tesla T4 با ۶هزار گام سیال و ۱۵هزار گام ذره آموزش دید. وزن‌ها داخل فایل‌اند و پیش‌فرض بارگذاری می‌شوند. برای تکرار آموزش TRAIN_FROM_SCRATCH=True کنید. زمان و GPU ثبت‌شده مربوط به آموزش اصلی است؛ آزمون‌ها با seed جدا و حل مستقل تکرار می‌شوند.

In [ ]:
from master9.train import train,load
TRAIN_FROM_SCRATCH=False  # GPU training already completed in version 1; set True to repeat.
if TRAIN_FROM_SCRATCH:
    model,runtime=train(OUTPUT,flow_steps=6000,particle_steps=15000,batch=2048,require_gpu=True)
else:
    import shutil
    OUTPUT.mkdir(exist_ok=True)
    for p in (ROOT/'pretrained').iterdir():shutil.copyfile(p,OUTPUT/p.name)
    model,runtime=load(OUTPUT)
print(json.dumps(runtime,indent=2))


## آزمون فیزیکی برای هر ورودی
شامل رینولدز و ویسکوزیتهٔ سازگار، پروفایل و دبی سیال، محل رهایش، سرعت اولیه و میرایی آن، تغییر چگالی، اعتبار درگ استوکس، باقیماندهٔ مستقل و مقایسه با حل stiff Radau است. شکست هر معیار اجرای موفق را متوقف می‌کند؛ آستانه‌ها از قبل در کد تعیین شده‌اند.

In [ ]:
from master9.validate import run
physics_report=run(OUTPUT,require_gpu=True)
print(json.dumps(physics_report,indent=2))


## دنبال‌کردن مسیر با پارامترهای جدید
کنترل به u=±0.5 محدود است. فرمان‌های حاصل در حل مستقل اینرسی‌دار بازپخش می‌شوند. چهار آزمایش شامل سرعت سیال و ویسکوزیتهٔ مرزی و رهایش متغیر است.

In [ ]:
from master9.train import load
from master9.control import demos
model,_=load(OUTPUT)
model=model.double()
control_report=demos(model,OUTPUT)
from IPython.display import Image,display
display(Image(filename=str(OUTPUT/'control_results.png')))


## استفاده با پارامتر و مسیر خود
در مثال زیر اعداد SI هستند. `from_re` ویسکوزیته را سازگار می‌سازد. برای مسیر دلخواه تابع `target(z)` در `master9/control.py` قابل تغییر است؛ z و x داخل کنترل بی‌بعدند. خروجی u جریان واقعی سیم‌پیچ نیست؛ مدل کالیبراسیون سخت‌افزار لازم است.

In [ ]:
from master9.physics import Parameters
p=Parameters.from_re(.8,mean=.015,rho=1000,x0=0,z0=-.08,vx0=.0005,vz0=.02)
print('Parameters:',p)
print('Re:',p.Re,'relaxation_s:',p.relaxation_s)
print('OUTPUTS:',OUTPUT)
print('ALL_PHYSICS_TESTS_PASSED:',physics_report['all_passed'])
print('ALL_CONTROL_TESTS_PASSED:',control_report['all_passed'])


## مسیر CSV دلخواه — اختیاری
عنوان فایل z_m,x_m است؛ همهٔ اعداد مترند. مسیر باید از نقطهٔ رهایش شروع شود، z صعودی داشته باشد و در دامنهٔ آموزش بماند. با فعال‌کردن RUN_MY_PATH می‌توانید نمونهٔ بسته را اجرا یا مسیر فایل خودتان را جایگزین کنید.

In [ ]:
RUN_MY_PATH=False
if RUN_MY_PATH:
    import numpy as np
    from master9.control import follow
    MY_PATH=ROOT/'examples/master9_target.csv'
    data,report=follow(model,p,target_csv=MY_PATH)
    np.savetxt(OUTPUT/'my_path_control.csv',data,delimiter=',',header='time_s,x_PINN_m,z_PINN_m,target_x_m,u,x_Radau_m,z_Radau_m,target_at_Radau_m',comments='')
    (OUTPUT/'my_path_report.json').write_text(json.dumps(report,indent=2))
    print(json.dumps(report,indent=2))
    if not report['passed']:raise RuntimeError('Requested path did not pass independent acceptance.')
